In [ ]:
!pip install imbalanced-learn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

from imblearn.over_sampling import SMOTE

In [ ]:
df = pd.read_csv('Vehicle_Maintenance_records.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'Vehicle_Maintenance_records.csv'

In [ ]:
df.head()

In [ ]:
for col in ["Last_Service_Date", "Warranty_Expiry_Date"]:
    if col in df.columns:
        df.drop(columns=col, inplace=True)

df = df.dropna(subset=["Need_Maintenance"])

for col in df.select_dtypes(include=np.number).columns:
    df[col] = df[col].fillna(df[col].median())

for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].fillna(df[col].mode()[0])

In [ ]:
# Remove extreme mileage
df = df[
    (df["Mileage"] > df["Mileage"].quantile(0.10)) &
    (df["Mileage"] < df["Mileage"].quantile(0.90))
]

# Remove extreme service history
df = df[
    (df["Service_History"] > df["Service_History"].quantile(0.10)) &
    (df["Service_History"] < df["Service_History"].quantile(0.90))
]

In [ ]:
df["mileage_per_year"] = df["Mileage"] / (df["Vehicle_Age"] + 1)

In [ ]:
df = pd.get_dummies(df, drop_first=True)

In [ ]:
X = df.drop("Need_Maintenance", axis=1)
y = df["Need_Maintenance"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print("After SMOTE:")
print(y_train_res.value_counts())

In [ ]:
scaler = StandardScaler()
X_train_res_scaled = scaler.fit_transform(X_train_res)
X_test_scaled = scaler.transform(X_test)

In [ ]:
log_model = LogisticRegression(max_iter=1000, C=0.3)
log_model.fit(X_train_res_scaled, y_train_res)

y_proba_log = log_model.predict_proba(X_test_scaled)[:,1]
y_pred_log = (y_proba_log > 0.8).astype(int)

print("=== Logistic Regression ===")
print("ROC-AUC:", roc_auc_score(y_test, y_proba_log))
print(confusion_matrix(y_test, y_pred_log))
print(classification_report(y_test, y_pred_log))

In [ ]:
plt.hist(y_proba_log, bins=30)
plt.title("Logistic Probability Distribution")
plt.show()

In [ ]:
tree_model = DecisionTreeClassifier(
    max_depth=6,
    min_samples_leaf=20,
    random_state=42
)

tree_model.fit(X_train_res, y_train_res)

y_proba_tree = tree_model.predict_proba(X_test)[:,1]
y_pred_tree = (y_proba_tree > 0.7).astype(int)

print("=== Decision Tree ===")
print("ROC-AUC:", roc_auc_score(y_test, y_proba_tree))
print(confusion_matrix(y_test, y_pred_tree))
print(classification_report(y_test, y_pred_tree))

In [ ]:
plt.hist(y_proba_tree, bins=30)
plt.title("Decision Tree Probability Distribution")
plt.show()

In [ ]:
import joblib

joblib.dump(log_model, "logistic_model_balanced.pkl")

In [ ]:
joblib.dump(tree_model, "decision_tree_balanced.pkl")

In [ ]:
joblib.dump(scaler, "scaler.pkl")

In [ ]:
joblib.dump(X.columns.tolist(), "feature_columns.pkl")

In [ ]:
from google.colab import files

files.download("logistic_model_balanced.pkl")
files.download("decision_tree_balanced.pkl")
files.download("scaler.pkl")
files.download("feature_columns.pkl")